# 🏠 Forecasting Household Energy Consumption
## Notebook 2 — Feature Engineering & Machine Learning

**Capstone Project | UMBC Data Science Master's Program**
Instructor: Dr. Chaojie (Jay) Wang | Author: Kushal Erramilli

---

### What This Notebook Does
Takes the clean hourly dataset from Notebook 1 and answers:

1. **Which features drive prediction?** — lag structure, rolling statistics, calendar
2. **Which model architecture works best?** — Baseline → Ridge → RF → XGBoost → LightGBM → LSTM
3. **How reliable are the predictions?** — walk-forward CV, error by hour and season
4. **What can a homeowner do with this?** — actionable interpretation of results

### Leakage Prevention Policy
All features represent information available *strictly before* the target hour `t`:

| Feature type | Correct | Incorrect (leakage) |
|---|---|---|
| Lag | `GAP[t-1]` → `shift(1)` | `GAP[t]` (no shift) |
| Rolling | `shift(1).rolling(w).mean()` | `.rolling(w).mean()` (includes t) |
| Diff | `shift(1).diff(k)` | `.diff(k)` without prior shift |

**Prerequisite:** Run `01_EDA.ipynb` first to generate `../data/hourly_clean.csv`.

### Notebook Roadmap
| Step | Section |
|------|---------|
| 1 | Load hourly dataset |
| 2 | Feature engineering (47 features, no leakage) |
| 3 | Train / Test split (80/20 temporal) |
| 4 | Evaluation metrics |
| 5 | Naïve baseline (lag-24h) |
| 6 | Ridge Regression |
| 7 | Random Forest |
| 8 | XGBoost |
| 9 | LightGBM |
| 10 | LSTM (Multivariate) |
| 11 | Model comparison |
| 12 | Feature importance — RF / XGB / LGB |
| 13 | Residual analysis |
| 14 | Walk-forward cross-validation |
| 15 | Error by hour and season |
| 16 | Quantile performance analysis |
| 17 | Save models |
| 18 | Final conclusions & recommendations |


## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

import warnings, joblib, os
warnings.filterwarnings("ignore")

SEED       = 42
TEST_RATIO = 0.20
TARGET     = "Global_active_power"
BAR_BLUE   = "#1565C0"
np.random.seed(SEED)
os.makedirs("../models", exist_ok=True)

print("✅ Imports OK")


import plotly.io as pio, plotly.graph_objects as go

_layout = dict(
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#2C2C2C"),
    title_font=dict(family="Inter, Arial, sans-serif", size=16, color="#0D2340", weight="bold"),
    paper_bgcolor="white", plot_bgcolor="#F7F9FC",
    colorway=["#1565C0","#E53935","#2E7D32","#F57C00","#6A1B9A","#00838F"],
    xaxis=dict(showgrid=True, gridcolor="#E8ECF0", gridwidth=1,
               zeroline=False, linecolor="#D0D5DD", linewidth=1.5,
               tickfont=dict(size=11), title_font=dict(size=12)),
    yaxis=dict(showgrid=True, gridcolor="#E8ECF0", gridwidth=1,
               zeroline=False, linecolor="#D0D5DD", linewidth=1.5,
               tickfont=dict(size=11), title_font=dict(size=12)),
    legend=dict(bgcolor="rgba(255,255,255,0.92)", bordercolor="#D0D5DD",
                borderwidth=1, font=dict(size=11)),
    margin=dict(l=65, r=45, t=75, b=65),
)
pio.templates["kushal"] = go.layout.Template(layout=_layout)
pio.templates.default = "kushal"
BAR_BLUE = "#1565C0"
print("✅ Professional theme applied.")


## 1. Load Hourly Dataset

**Why hourly?** Minute-level data (2 M rows) is computationally expensive
and introduces high-frequency noise that adds no forecasting value for
hour-ahead predictions. Hourly aggregation smooths sensor jitter while
preserving every meaningful temporal pattern identified in Notebook 1.


In [ ]:
df = pd.read_csv("../data/hourly_clean.csv",
                 index_col="Datetime", parse_dates=True)
df.sort_index(inplace=True)

print(f"Shape      : {df.shape}")
print(f"Date range : {df.index.min()}  →  {df.index.max()}")
print(f"Missing    : {df.isnull().sum().sum()}")
df.describe().T


## 2. Feature Engineering

We create **47 features** across five categories. Every feature respects
the leakage policy — no information from hour `t` leaks into the inputs.

### Why each group matters

| Category | Features | Purpose |
|----------|---------|---------|
| **Calendar** | hour, dow, month, quarter, weekend | Captures routine human behaviour cycles |
| **Cyclical** | sin/cos of hour, dow, month | Prevents discontinuity at midnight, Sunday→Monday, Dec→Jan |
| **Lag** | 1, 2, 3, 6, 12, 24, 48, 72, 168, 336 h | Autocorrelation structure from EDA (ACF significant to 2 weeks) |
| **Rolling** | mean, std, min, max over 3, 6, 12, 24, 48, 168 h | Smoothed trend + volatility signal |
| **Diff** | 1 h and 24 h change | Rate of change — is consumption accelerating? |

**Sanity check:** After engineering we verify that no feature has correlation
|r| > 0.98 with the target (would indicate leakage).


In [ ]:
df_feat = df[[TARGET]].copy()

# ── Calendar & cyclical ───────────────────────────────────────────────────
def add_time_features(d):
    d["hour"]      = d.index.hour
    d["dayofweek"] = d.index.dayofweek
    d["month"]     = d.index.month
    d["quarter"]   = d.index.quarter
    d["is_weekend"]= (d.index.dayofweek >= 5).astype(int)
    d["hour_sin"]  = np.sin(2*np.pi*d["hour"]      / 24)
    d["hour_cos"]  = np.cos(2*np.pi*d["hour"]      / 24)
    d["dow_sin"]   = np.sin(2*np.pi*d["dayofweek"] / 7)
    d["dow_cos"]   = np.cos(2*np.pi*d["dayofweek"] / 7)
    d["month_sin"] = np.sin(2*np.pi*d["month"]     / 12)
    d["month_cos"] = np.cos(2*np.pi*d["month"]     / 12)
    return d

# ── Lag (all shift ≥ 1) ──────────────────────────────────────────────────
def add_lag_features(d, lags=[1,2,3,6,12,24,48,72,168,336]):
    for lag in lags:
        d[f"lag_{lag}h"] = d[TARGET].shift(lag)
    return d

# ── Rolling (shift(1) before rolling) ────────────────────────────────────
def add_rolling_features(d, windows=[3,6,12,24,48,168]):
    past = d[TARGET].shift(1)
    for w in windows:
        r = past.rolling(w)
        d[f"roll_mean_{w}h"] = r.mean()
        d[f"roll_std_{w}h"]  = r.std()
        d[f"roll_min_{w}h"]  = r.min()
        d[f"roll_max_{w}h"]  = r.max()
    return d

# ── Diff (shift(1) FIRST — past-to-past change only) ─────────────────────
def add_diff_features(d):
    past = d[TARGET].shift(1)
    d["diff_1h"]  = past.diff(1)
    d["diff_24h"] = past.diff(24)
    return d

df_feat = add_time_features(df_feat)
df_feat = add_lag_features(df_feat)
df_feat = add_rolling_features(df_feat)
df_feat = add_diff_features(df_feat)
df_feat.dropna(inplace=True)

feature_cols = [c for c in df_feat.columns if c != TARGET]
print(f"Rows after dropna : {len(df_feat):,}")
print(f"Feature count     : {len(feature_cols)}")


In [ ]:
# Leakage sanity check
corr_target = df_feat.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("Top 5 feature correlations with target (must all be < 0.98):")
print(corr_target.head())
assert corr_target.max() < 0.98, "⚠️  Leakage detected!"
print(f"\n✅ Leakage check passed — max |r| = {corr_target.max():.4f}")
print(f"   Top predictor: lag_1h (r={corr_target['lag_1h']:.4f})")


## 3. Train / Test Split — 80 / 20 Temporal

**Why temporal (not random) split?**
Random splitting would mix future data into training — a form of leakage.
A temporal split ensures the model is evaluated on data it has *never seen*
chronologically, mimicking real deployment conditions.

**What this means:**
- **Train:** Dec 2006 → Feb 2010 (~3.1 years)
- **Test:**  Feb 2010 → Nov 2010 (~9 months)

The test set spans all four seasons, so seasonal performance differences
measured in Section 15 are meaningful.


In [ ]:
X = df_feat[feature_cols]
y = df_feat[TARGET]

split_idx = int(len(df_feat) * (1 - TEST_RATIO))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]:,} hours  ({X_train.index.min().date()} → {X_train.index.max().date()})")
print(f"Test : {X_test.shape[0]:,} hours   ({X_test.index.min().date()} → {X_test.index.max().date()})")

# Weekly resampled for cleaner area plot
y_train_w = y_train.resample("W").mean()
y_test_w  = y_test.resample("W").mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_train.index, y=y_train,
    name="Train", line=dict(color="#1565C0", width=0.7),
    fill="tozeroy", fillcolor="rgba(21,101,192,0.18)",
))
fig.add_trace(go.Scatter(
    x=y_test.index, y=y_test,
    name="Test", line=dict(color="#C62828", width=0.7),
    fill="tozeroy", fillcolor="rgba(198,40,40,0.18)",
))
# Vertical split line
split_date = y_test.index.min()
fig.add_vline(
    x=str(split_date),
    line_dash="dash",
    line_color="#555",
    line_width=1.4,
)
fig.update_layout(
    title="Temporal Train / Test Split — Test Covers All Four Seasons of 2010",
    yaxis_title="Avg Power (kW)", title_x=0.5, height=410,
    xaxis=dict(title=""),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
)
fig.show()

## 4. Evaluation Metrics

We report four metrics because they capture different aspects of error:

| Metric | Formula | Why |
|--------|---------|-----|
| **RMSE** | √(mean((y−ŷ)²)) | Penalises large errors heavily; critical for peak events |
| **MAE** | mean(|y−ŷ|) | Average absolute error in kW; interpretable in real units |
| **MAPE** | mean(|y−ŷ|/y)×100 | Percentage error; useful for communicating to non-technical audience |
| **R²** | 1 − SS_res/SS_tot | Fraction of variance explained; 1.0 = perfect, 0 = no better than mean |

A model that only predicts the mean of the training set would score R² = 0.
The naïve baseline scoring R² = −0.14 means it is *worse* than predicting
the mean every hour.


In [ ]:
results = []

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2   = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-y_true.mean())**2)
    print(f"[{name:<22}]  RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"MAPE={mape:.1f}%  R²={r2:.4f}")
    return {"Model":name,"RMSE":round(rmse,4),"MAE":round(mae,4),
            "MAPE(%)":round(mape,2),"R2":round(r2,4)}


## 5. Naïve Baseline — Lag-24h Persistence

**What it does:** Predicts this hour's usage as the same hour yesterday.
This is the simplest meaningful baseline — it exploits only the daily cycle.

**Why we need it:** Any model that cannot beat this "same time yesterday"
baseline provides zero value over a simple spreadsheet lookup.

**Result (from actual run) — RMSE: 0.778, MAE: 0.520, R²: −0.140**

The **negative R²** means the naïve predictor is *worse* than always predicting the training mean. It fails because actual usage varies day-to-day (weather, occupancy, events) in ways that yesterday's reading cannot capture. This gap — 0.778 vs the best model's 0.459 — is what 47 engineered features and gradient boosting buy us: a **41% RMSE reduction**.

In [ ]:
y_pred_naive = X_test["lag_24h"].values
results.append(evaluate("Naive (lag 24h)", y_test.values, y_pred_naive))

N = 500
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=y_test.values[:N], name="Actual",
    line=dict(color="#1565C0", width=1.5),
))
fig.add_trace(go.Scatter(
    y=y_pred_naive[:N], name="Naïve Forecast (lag-24h)",
    line=dict(color="#F57C00", width=1.5, dash="dash"),
))
fig.update_layout(
    title="Naïve Baseline — Systematically Wrong at Peaks and Troughs",
    yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
    title_x=0.5, height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

## 6. Ridge Regression

**What it does:** Linear regression with L2 penalty to prevent overfitting. All 47 features are scaled to zero-mean unit-variance before fitting.

**Why Ridge before tree models?** Ridge is interpretable and fast. Its performance sets an upper bound for linear approaches.

**Hyperparameters:** `alpha=0.5` (mild regularisation); StandardScaler pipeline for stable coefficient estimation.

**Result (from actual run) — RMSE: 0.495, MAE: 0.349, R²: 0.538**

A **36% RMSE reduction** vs the naïve baseline — substantial. Ridge extracts real signal from the lag features (lag_1h dominates linearly). However, it fails to capture nonlinear interactions like *"it's winter AND an evening peak AND the last hour was already high"* — that requires tree-based models.

In [ ]:
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  Ridge(alpha=0.5))
])
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)
results.append(evaluate("Ridge Regression", y_test.values, y_pred_ridge))

N = 500
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:N], name="Actual",
    line=dict(color="#1565C0", width=1.5)))
fig.add_trace(go.Scatter(y=y_pred_ridge[:N], name="Ridge Forecast",
    line=dict(color="#2E7D32", width=1.5, dash="dash")))
fig.update_layout(
    title="Ridge Regression — Tracks Trend Well, Misses Sharp Spikes",
    yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
    title_x=0.5, height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()
joblib.dump(ridge_pipe, "../models/ridge_pipeline.pkl")
print("✅ Ridge model saved")

## 7. Random Forest

**What it does:** Ensemble of 300 decision trees, each trained on a random subset of features. Predictions are averaged to reduce variance.

**Why Random Forest?** Handles nonlinear interactions naturally, robust to outliers, and feature importances are easy to interpret.

**Hyperparameters:**
- `n_estimators=300` — enough trees for stable importance scores
- `max_depth=20` — deep enough to capture interactions
- `max_features=0.6` — 60% of features per split (decorrelates trees, reduces variance)

**Result (from actual run) — RMSE: 0.461, MAE: 0.316, R²: 0.599**

Reduces RMSE by another 7% vs Ridge. The forest correctly captures that *lag_1h × hour × season* interact nonlinearly — e.g., a high lag_1h reading at 21h in winter reliably predicts high consumption in the next hour.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300, max_depth=20, min_samples_leaf=3,
    max_features=0.6, n_jobs=-1, random_state=SEED
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(evaluate("Random Forest", y_test.values, y_pred_rf))

N = 500
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:N], name="Actual",
    line=dict(color="#1565C0", width=1.5)))
fig.add_trace(go.Scatter(y=y_pred_rf[:N], name="RF Forecast",
    line=dict(color="#6A1B9A", width=1.5, dash="dash")))
fig.update_layout(
    title="Random Forest — Captures Shape Well, Slight Smoothing at Extreme Peaks",
    yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
    title_x=0.5, height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()
joblib.dump(rf, "../models/random_forest.pkl")
print("✅ Random Forest saved")

## 8. XGBoost

**What it does:** Sequential gradient boosting — each tree corrects the residuals of all previous trees. Early stopping prevents overfitting.

**Why XGBoost over Random Forest?** Boosting focuses learning on hard examples (residual errors), making it better at capturing infrequent peak events.

**Hyperparameters:**
- `learning_rate=0.03` + `n_estimators=1000` with early stopping — slow learning + many iterations is the standard high-performance XGB recipe
- `reg_alpha=0.05, reg_lambda=1.5` — L1 + L2 regularisation

**Result (from actual run) — RMSE: 0.459, MAE: 0.316, R²: 0.602**

Marginal improvement over Random Forest (< 0.5% RMSE reduction). The near-tie tells us we are approaching the **noise floor** of this dataset — the remaining ~0.46 kW RMSE is largely irreducible occupancy and weather variation, not a modelling deficiency.

In [ ]:
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=7,
        min_child_weight=3, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.05, reg_lambda=1.5,
        early_stopping_rounds=30, random_state=SEED, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_xgb = xgb_model.predict(X_test)
    results.append(evaluate("XGBoost", y_test.values, y_pred_xgb))

    N = 500
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_test.values[:N], name="Actual",
        line=dict(color="#1565C0", width=1.5)))
    fig.add_trace(go.Scatter(y=y_pred_xgb[:N], name="XGBoost Forecast",
        line=dict(color="#C62828", width=1.5, dash="dash")))
    fig.update_layout(
        title="XGBoost — Closely Tracks Actual, Tighter Peak Estimates Than Ridge",
        yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
        title_x=0.5, height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()
    joblib.dump(xgb_model, "../models/xgboost_model.pkl")
    print("✅ XGBoost saved")
except ImportError:
    print("pip install xgboost")

## 9. LightGBM ← Best Model

**What it does:** Leaf-wise gradient boosting with histogram-based split finding. Faster and often more accurate than XGBoost on tabular data.

**Why LightGBM wins:**
- Leaf-wise growth captures deeper, more specific decision boundaries with fewer iterations
- Histogram binning handles the 27k training rows efficiently without memory pressure
- Slightly higher R² than XGBoost due to better exploitation of fine-grained lag patterns

**Result (from actual run) — RMSE: 0.459, MAE: 0.314, R²: 0.603 ← Best overall**

The improvement over naïve (0.778 → 0.459) represents a **41% RMSE reduction**. Predictions are within ±0.314 kW on average (MAE) — good enough to identify peak windows and recommend load-shifting.

**Why not perfect?** R² = 0.603 means ~40% of variance remains unexplained. This is expected — *occupancy* (is someone home? what are they cooking?) is the dominant unobserved driver and is absent from this dataset.

In [ ]:
try:
    import lightgbm as lgb
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, num_leaves=63, max_depth=8,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.05, reg_lambda=1.5, random_state=SEED, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(
        X_train, y_train, eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
    )
    y_pred_lgb = lgb_model.predict(X_test)
    results.append(evaluate("LightGBM", y_test.values, y_pred_lgb))

    N = 500
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_test.values[:N], name="Actual",
        line=dict(color="#1565C0", width=1.5)))
    fig.add_trace(go.Scatter(y=y_pred_lgb[:N], name="LightGBM Forecast",
        line=dict(color="#00695C", width=1.5, dash="dash")))
    fig.update_layout(
        title="LightGBM (Best Model) — Forecast Closely Follows Actual Usage Pattern",
        yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
        title_x=0.5, height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()
    joblib.dump(lgb_model, "../models/lightgbm_model.pkl")
    print("✅ LightGBM saved")
except ImportError:
    print("pip install lightgbm")

## 10. LSTM — Multivariate

**What it does:** A three-layer LSTM network reads 48 consecutive hours of all 47 features and predicts the next hour.

**Architecture:**
- Layer 1: LSTM(128) + Dropout(0.2)
- Layer 2: LSTM(64) + Dropout(0.2)
- Layer 3: LSTM(32) + Dropout(0.1)
- Dense(16, ReLU) → Dense(1)  |  Total parameters: ~152,000

**Result (from actual run) — RMSE: 0.584, MAE: 0.422, R²: 0.357 ← Worst ML model**

**Why does LSTM underperform all tree models here?**
The loss curve (see chart below) shows validation loss diverges from training loss after epoch ~3 — a classic overfitting signature. Root causes:
1. **47 engineered features already encode all temporal structure** — lag and rolling features give tree models the same sequence information the LSTM is trying to learn, but more efficiently
2. **Only ~27k training sequences** — deep sequence models typically need 10–100× more data to generalise
3. **No exogenous signals** (weather, calendar events) — LSTMs excel when external time-varying drivers modulate the target; here there are none

**Decision:** Deploy LightGBM. The LSTM result is an important negative finding — deep learning is not automatically superior for tabular time-series forecasting.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.random.set_seed(SEED)
    LOOKBACK = 48

    scaler_lstm = StandardScaler()
    X_lstm_all  = scaler_lstm.fit_transform(df_feat[feature_cols])
    y_lstm_all  = df_feat[TARGET].values

    def make_sequences(X, y, lookback):
        Xs, ys = [], []
        for i in range(lookback, len(X)):
            Xs.append(X[i-lookback:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    X_seq, y_seq = make_sequences(X_lstm_all, y_lstm_all, LOOKBACK)
    split_lstm   = int(len(X_seq) * (1 - TEST_RATIO))
    X_tr, X_te   = X_seq[:split_lstm], X_seq[split_lstm:]
    y_tr, y_te   = y_seq[:split_lstm], y_seq[split_lstm:]

    n_feat = X_seq.shape[2]
    lstm_model = keras.Sequential([
        layers.LSTM(128, input_shape=(LOOKBACK, n_feat), return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.1),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ])
    lstm_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    lstm_model.summary()

    cb_es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    cb_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=0)

    history = lstm_model.fit(
        X_tr, y_tr, epochs=50, batch_size=256,
        validation_split=0.1, callbacks=[cb_es, cb_lr], verbose=1
    )

    y_pred_lstm = lstm_model.predict(X_te).flatten()
    results.append(evaluate("LSTM (multivariate)", y_te, y_pred_lstm))

    # ── Loss curve ───────────────────────────────────────────────────────────
    loss_df = pd.DataFrame({
        "Train Loss": history.history["loss"],
        "Val Loss":   history.history["val_loss"]
    }).reset_index().rename(columns={"index": "Epoch"})
    loss_melt = loss_df.melt(id_vars="Epoch", var_name="Series", value_name="MSE Loss")

    fig_loss = px.line(
        loss_melt, x="Epoch", y="MSE Loss", color="Series",
        title="LSTM Training — Val Loss Diverges After Epoch ~3 (Overfitting)",
        color_discrete_map={"Train Loss": "#1565C0", "Val Loss": "#E53935"},
    )
    fig_loss.update_traces(line=dict(width=2.2))
    fig_loss.update_layout(
        title_x=0.5, height=380,
        yaxis_title="MSE Loss",
        legend=dict(title="", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig_loss.show()

    # ── Forecast vs actual ───────────────────────────────────────────────────
    N = 500
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_te[:N], name="Actual",
        line=dict(color="#1565C0", width=1.5)))
    fig.add_trace(go.Scatter(y=y_pred_lstm[:N], name="LSTM Forecast",
        line=dict(color="#F57C00", width=1.5, dash="dash")))
    fig.update_layout(
        title="LSTM — Over-Smoothed; Misses Sharp Peaks That Tree Models Capture",
        yaxis_title="Avg Power (kW)", xaxis_title="Test Hours (first 500)",
        title_x=0.5, height=400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()
    lstm_model.save("../models/lstm_model.keras")
    print("✅ LSTM saved")
except ImportError:
    print("pip install tensorflow")

## 11. Model Comparison

**All six models sorted by RMSE (from actual run output):**

| Model | RMSE ↓ | MAE ↓ | R² ↑ |
|-------|--------|-------|------|
| **LightGBM** ⭐ | **0.459** | **0.314** | **0.603** |
| XGBoost | 0.459 | 0.316 | 0.602 |
| Random Forest | 0.461 | 0.316 | 0.599 |
| Ridge Regression | 0.495 | 0.349 | 0.538 |
| LSTM (multivariate) | 0.584 | 0.422 | 0.357 |
| Naïve (lag-24h) | 0.778 | 0.520 | −0.140 |

**Key takeaways:**
- **LightGBM, XGBoost, Random Forest are statistically indistinguishable** (< 0.5% RMSE gap) — they have reached the dataset noise floor together
- **Ridge vs tree models: 8% RMSE gap** confirms nonlinear feature interactions are material
- **LSTM is the worst ML model** (RMSE 0.584, R² 0.357) — deep learning is not superior here
- **Deployment choice: LightGBM** — fastest inference, best RMSE, native feature importance

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE")
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))


In [ ]:
# Read actual results from the results list
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

MODEL_COLORS = {
    "LightGBM":           "#00695C",
    "XGBoost":            "#C62828",
    "Random Forest":      "#6A1B9A",
    "Ridge Regression":   "#2E7D32",
    "LSTM (multivariate)":"#E65100",
    "Naive (lag 24h)":    "#78909C",
}
bar_colors = [MODEL_COLORS.get(m, "#1565C0") for m in results_df["Model"]]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["RMSE  (lower = better)", "MAE  (lower = better)", "R²  (higher = better)"],
    horizontal_spacing=0.08,
)

for col, metric in enumerate(["RMSE","MAE","R2"], 1):
    vals = results_df[metric]
    is_best = (vals == vals.min()) if metric != "R2" else (vals == vals.max())
    bc = ["#FFD600" if b else c for b, c in zip(is_best, bar_colors)]
    fig.add_trace(
        go.Bar(
            x=results_df["Model"], y=vals,
            marker_color=bc,
            marker_line_color=["#333" if b else "rgba(0,0,0,0.1)" for b in is_best],
            marker_line_width=[2 if b else 0.8 for b in is_best],
            text=[f"{v:.3f}" for v in vals],
            textposition="outside",
            textfont=dict(size=10),
            showlegend=False,
        ),
        row=1, col=col
    )

fig.update_layout(
    title="Model Performance Comparison — LightGBM Wins (Highlighted in Gold)",
    title_x=0.5, height=540,
    font=dict(size=10),
    paper_bgcolor="white", plot_bgcolor="#F7F9FC",
)
for i in range(1, 4):
    fig.update_xaxes(tickangle=-30, tickfont=dict(size=9), row=1, col=i)
    fig.update_yaxes(showgrid=True, gridcolor="#E8ECF0", row=1, col=i)

fig.show()

## 12. Feature Importance

**Why:** Feature importances tell us which signals matter most — directly informing the Streamlit app's energy advisor and guiding future feature development.

### Consistent findings across all three tree models (from actual importance charts):

1. **`lag_1h` dominates massively** (RF: 0.424 importance, XGB: 0.300, LGB: 783 splits) — the last hour of usage is far and away the strongest predictor. The ratio of lag_1h to the next feature is >5× in RF.

2. **`diff_1h` ranks 2nd in Random Forest** (0.046) — rate of change matters: is consumption rising or falling right now?

3. **`roll_mean_3h` / `roll_min_3h`** rank highly across all models — very recent rolling context (< 3 hours) is more informative than long-horizon rolling features

4. **`lag_168h` (same time last week)** ranks in top 6 for LightGBM — weekly routine is a stronger signal than most rolling features

5. **`hour` ranks 2nd in LightGBM** (688 splits) — calendar time-of-day has outsized importance in the leaf-wise model which explores deeper interactions

6. **Calendar features** (`hour`, `hour_sin`, `hour_cos`, `dayofweek`) all appear in top 15 across models — confirming the strong intraday cycle from EDA

**Homeowner implication:** *Your consumption right now is the best predictor of the next hour.* Cutting peak loads immediately compounds — a high lag_1h leads the model to predict continued high usage.

In [ ]:
# Random Forest importance
rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns)
top_rf = rf_imp.nlargest(20).sort_values()

# Color: top 1 gold, top 2-5 deep blue, rest lighter
n = len(top_rf)
rf_colors = ["#1565C0"] * n
rf_colors[-1] = "#FFD600"          # lag_1h = gold
rf_colors[-2] = "#1E88E5"          # 2nd
rf_colors[-3] = "#1E88E5"          # 3rd
rf_colors[-4] = "#1E88E5"          # 4th
rf_colors[-5] = "#1E88E5"          # 5th

fig = go.Figure(go.Bar(
    x=top_rf.values, y=top_rf.index, orientation="h",
    marker_color=rf_colors,
    marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.6,
    text=[f"{v:.4f}" for v in top_rf.values],
    textposition="outside",
    textfont=dict(size=9),
))
fig.update_layout(
    title="Top 20 Feature Importances — Random Forest  (lag_1h dominates at 0.424)",
    xaxis_title="Importance Score (MDI)", title_x=0.5, height=600,
    yaxis=dict(tickfont=dict(size=10)),
    xaxis=dict(range=[0, top_rf.max() * 1.2]),
)
fig.show()

In [ ]:
# XGBoost importance
try:
    xgb_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
    top_xgb = xgb_imp.nlargest(20).sort_values()
    n = len(top_xgb)
    xc = ["#C62828"] * n
    xc[-1] = "#FFD600"
    xc[-2] = "#EF9A9A"
    xc[-3] = "#EF9A9A"
    xc[-4] = "#EF9A9A"
    xc[-5] = "#EF9A9A"

    fig = go.Figure(go.Bar(
        x=top_xgb.values, y=top_xgb.index, orientation="h",
        marker_color=xc,
        marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.6,
        text=[f"{v:.4f}" for v in top_xgb.values],
        textposition="outside", textfont=dict(size=9),
    ))
    fig.update_layout(
        title="Top 20 Feature Importances — XGBoost  (lag_1h=0.300, roll_mean_3h=0.064)",
        xaxis_title="Importance Score (F-score)", title_x=0.5, height=600,
        yaxis=dict(tickfont=dict(size=10)),
        xaxis=dict(range=[0, top_xgb.max() * 1.2]),
    )
    fig.show()
except NameError: pass

In [ ]:
# LightGBM importance
try:
    lgb_imp = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
    top_lgb = lgb_imp.nlargest(20).sort_values()
    n = len(top_lgb)
    lc = ["#00695C"] * n
    lc[-1] = "#FFD600"
    lc[-2] = "#4DB6AC"
    lc[-3] = "#4DB6AC"
    lc[-4] = "#4DB6AC"
    lc[-5] = "#4DB6AC"

    fig = go.Figure(go.Bar(
        x=top_lgb.values, y=top_lgb.index, orientation="h",
        marker_color=lc,
        marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.6,
        text=[f"{v:.0f}" for v in top_lgb.values],
        textposition="outside", textfont=dict(size=9),
    ))
    fig.update_layout(
        title="Top 20 Feature Importances — LightGBM  (lag_1h=783, hour=688, lag_168h=674 splits)",
        xaxis_title="Split Count", title_x=0.5, height=600,
        yaxis=dict(tickfont=dict(size=10)),
        xaxis=dict(range=[0, top_lgb.max() * 1.22]),
    )
    fig.show()
except NameError: pass

## 13. Residual Analysis — LightGBM

**Why:** Residuals (actual − predicted) reveal *systematic errors*. Patterns in residuals mean the model is still missing some structure.

**What we find (from actual residual plots):**
- **Residual distribution:** Approximately bell-shaped and centred near zero — no systematic bias
- **Slight right tail:** The model under-predicts extreme high values (> 4 kW) — rare multi-appliance coincidence events are genuinely hard to predict
- **Actual vs Predicted scatter:** Points cluster near the perfect-prediction diagonal, but scatter fans out at higher actuals — classic heteroscedasticity (larger errors at larger loads), which is typical and expected for energy forecasting
- **80%+ of predictions fall within ±0.5 kW** — operationally useful accuracy for scheduling purposes
- **Conclusion:** The model is not systematically biased. Residual structure suggests remaining error is close to irreducible noise without additional data (weather, occupancy).

In [ ]:
try:
    best_pred = y_pred_lgb; best_name = "LightGBM"
except NameError:
    try: best_pred = y_pred_xgb; best_name = "XGBoost"
    except NameError: best_pred = y_pred_rf; best_name = "Random Forest"

residuals = y_test.values - best_pred
within_05 = (np.abs(residuals) <= 0.5).mean() * 100
within_10 = (np.abs(residuals) <= 1.0).mean() * 100
print(f"Bias (mean residual)     : {residuals.mean():.4f} kW")
print(f"Std of residuals         : {residuals.std():.4f} kW")
print(f"Within ±0.5 kW           : {within_05:.1f}%")
print(f"Within ±1.0 kW           : {within_10:.1f}%")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Residual Distribution — Centred at {residuals.mean():.3f} kW (Near-Zero Bias)",
        "Actual vs Predicted — Scatter Fans at High Loads (Heteroscedasticity)",
    ],
    horizontal_spacing=0.1,
)
# Left: residual histogram
fig.add_trace(go.Histogram(
    x=residuals, nbinsx=90,
    marker_color="#1565C0", marker_line_color="white", marker_line_width=0.4,
    opacity=0.85,
), row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="#E53935", line_width=1.6, row=1, col=1)

# Right: actual vs predicted scatter
samp = np.random.choice(len(y_test), min(2500, len(y_test)), replace=False)
fig.add_trace(go.Scatter(
    x=y_test.values[samp], y=best_pred[samp],
    mode="markers",
    marker=dict(color="#1565C0", size=3.5, opacity=0.35,
                line=dict(color="white", width=0.3)),
    showlegend=False,
), row=1, col=2)
mn, mx = float(y_test.min()), float(y_test.max())
fig.add_trace(go.Scatter(
    x=[mn, mx], y=[mn, mx], mode="lines",
    line=dict(color="#E53935", dash="dash", width=1.8),
    name="Perfect prediction", showlegend=False,
), row=1, col=2)

fig.update_layout(
    title=f"{best_name} — Residual Analysis  |  Within ±0.5 kW: {within_05:.1f}%",
    title_x=0.5, height=490, showlegend=False,
    paper_bgcolor="white", plot_bgcolor="#F7F9FC",
)
fig.update_xaxes(title_text="Residual (kW)", showgrid=True, gridcolor="#E8ECF0", row=1, col=1)
fig.update_xaxes(title_text="Actual (kW)",   showgrid=True, gridcolor="#E8ECF0", row=1, col=2)
fig.update_yaxes(title_text="Count",         showgrid=True, gridcolor="#E8ECF0", row=1, col=1)
fig.update_yaxes(title_text="Predicted (kW)",showgrid=True, gridcolor="#E8ECF0", row=1, col=2)
fig.show()

## 14. Walk-Forward Cross-Validation

**Why:** A single train/test split could be unusually easy or hard. Walk-forward CV mimics sequential deployment — each fold trains on all past data and tests on the next chronological block.

**What we find (from actual CV run output):**
| Fold | RMSE | Note |
|------|------|------|
| 1 | 0.636 | Hardest — model trained on least data |
| 2 | 0.520 | Stabilises |
| 3 | 0.516 | Stable |
| 4 | 0.521 | Stable |
| 5 | 0.452 | Easiest — most training data, summer test period |
| **Mean ± Std** | **0.529 ± 0.061** | Low variance confirms temporal stability |

The low Std (0.061) confirms the model **does not collapse or overfit on any specific time period**. The monotonically improving trend (fold 1 → fold 5) shows the model benefits consistently from more training data.

In [ ]:
tscv   = TimeSeriesSplit(n_splits=5)
rf_cv  = RandomForestRegressor(n_estimators=100, max_depth=15,
                                min_samples_leaf=3, max_features=0.6,
                                n_jobs=-1, random_state=SEED)
cv_raw  = cross_val_score(rf_cv, X, y, cv=tscv,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_rmse = -cv_raw

print("Walk-Forward CV RMSE per fold:")
for i,s in enumerate(cv_rmse, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean ± Std : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")

fold_labels = [f"Fold {i}" for i in range(1,6)]
bar_colors  = ["#78909C" if v > cv_rmse.mean() else "#1565C0" for v in cv_rmse]
bar_colors[0] = "#C62828"   # Hardest fold highlighted

fig = go.Figure(go.Bar(
    x=fold_labels, y=cv_rmse,
    marker_color=bar_colors,
    marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.8,
    text=[f"{v:.4f}" for v in cv_rmse],
    textposition="outside",
    textfont=dict(size=11),
))
fig.add_hline(
    y=cv_rmse.mean(), line_dash="dash", line_color="#E53935", line_width=1.8,
    annotation_text=f"Mean = {cv_rmse.mean():.4f} kW  (Std = {cv_rmse.std():.4f})",
    annotation_font=dict(size=11, color="#E53935"),
    annotation_position="top right",
)
fig.update_layout(
    title="Walk-Forward Cross-Validation RMSE — Stable Across Time (Low Std = 0.061)",
    title_x=0.5, height=430,
    xaxis_title="Fold (Chronological)", yaxis_title="RMSE (kW)",
    yaxis=dict(range=[0, cv_rmse.max() * 1.18]),
    annotations=[dict(
        text="Red = hardest (least training data)  |  Blue = stable folds  |  Dark = above mean",
        x=0.5, y=1.09, xref="paper", yref="paper",
        showarrow=False, font=dict(size=10, color="#555"), xanchor="center",
    )],
)
fig.show()

## 15. Error Analysis — By Hour and Season

**Why:** Knowing *when* the model is least accurate tells homeowners which forecasts to trust and helps identify the highest-value targets for future improvement.

### By Hour (from actual MAE chart):
- **Night floor (hours 3–4): MAE ≈ 0.13–0.14 kW** — lowest error window; standby loads are stable and predictable
- **Hour 0 (midnight): MAE ≈ 0.20 kW** — slightly higher as the model transitions from evening
- **Hour 6 (6 AM): MAE spikes to ≈ 0.38 kW** — sharp morning ramp-up from standby to active appliances; highly variable day-to-day
- **Hours 7–12: MAE ≈ 0.31–0.36 kW** — elevated but settles as the morning routine stabilises
- **Hours 19–22: MAE = 0.41–0.49 kW — hardest window overall**; dinner + TV + arbitrary appliance combinations produce the highest and most variable loads
- **Hour 22 is the single highest-error hour at ≈ 0.49 kW** — late-evening activity is the least predictable from historical patterns

### By Season (from actual MAE chart — actual values, not assumed):
- **Summer: MAE = 0.265 kW** — easiest; AC-driven load follows a predictable temperature pattern
- **Spring: MAE = 0.322 kW**
- **Autumn: MAE = 0.346 kW**
- **Winter: MAE = 0.361 kW — hardest**; heating behaviour varies more day-to-day (thermostat settings, cooking, occupancy)

**Winter is ~36% harder to predict than summer** (0.361 / 0.265 = 1.36×).

**Homeowner implication:** Treat evening (19–22h) and winter forecasts with greater uncertainty. Night-time (2–5 AM) forecasts are highly reliable.

In [ ]:
error_df = pd.DataFrame({
    "actual"  : y_test.values,
    "pred"    : best_pred,
    "abs_err" : np.abs(y_test.values - best_pred),
    "hour"    : y_test.index.hour,
    "season"  : y_test.index.month.map({
        12:"Winter",1:"Winter",2:"Winter",
        3:"Spring", 4:"Spring",5:"Spring",
        6:"Summer", 7:"Summer",8:"Summer",
        9:"Autumn",10:"Autumn",11:"Autumn"})
}, index=y_test.index)

# Hour of day
h_err = error_df.groupby("hour")["abs_err"].mean().reset_index()
h_err.columns = ["Hour","MAE"]
max_hr = h_err.loc[h_err["MAE"].idxmax(), "Hour"]

bar_cols_h = []
for h in h_err["Hour"]:
    if h == max_hr:           bar_cols_h.append("#C62828")  # peak error
    elif 3 <= h <= 5:         bar_cols_h.append("#2E7D32")  # safest
    elif 19 <= h <= 22:       bar_cols_h.append("#E53935")  # hard zone
    else:                     bar_cols_h.append("#1565C0")

fig = go.Figure(go.Bar(
    x=h_err["Hour"], y=h_err["MAE"],
    marker_color=bar_cols_h,
    marker_line_color="rgba(0,0,0,0.08)", marker_line_width=0.5,
))
fig.add_annotation(x=max_hr, y=h_err["MAE"].max() + 0.015,
    text=f"Peak: {h_err['MAE'].max():.3f} kW at {max_hr}h",
    showarrow=False, font=dict(size=10, color="#C62828"))
fig.update_layout(
    title=f"MAE by Hour of Day — Night Floor 0.13 kW (hr 3–4), Evening Peak 0.49 kW (hr {max_hr})",
    title_x=0.5, height=430,
    xaxis=dict(title="Hour of Day", tickmode="linear", dtick=2),
    yaxis=dict(title="MAE (kW)", range=[0, h_err["MAE"].max() * 1.18]),
    annotations=[dict(
        text="Green = easiest (night)  |  Blue = moderate  |  Red = hardest (evenings)",
        x=0.5, y=1.09, xref="paper", yref="paper",
        showarrow=False, font=dict(size=10, color="#555"), xanchor="center",
    )],
)
fig.show()

In [ ]:
s_err = (error_df.groupby("season")["abs_err"].mean()
         .reindex(["Winter","Spring","Summer","Autumn"]).reset_index())
s_err.columns = ["Season","MAE"]

season_colors = {"Winter":"#1565C0","Spring":"#2E7D32","Summer":"#F57C00","Autumn":"#6A1B9A"}
sc = [season_colors[s] for s in s_err["Season"]]

pct_harder = (s_err.loc[s_err.Season=="Winter","MAE"].values[0] /
              s_err.loc[s_err.Season=="Summer","MAE"].values[0] - 1) * 100

fig = go.Figure(go.Bar(
    x=s_err["Season"], y=s_err["MAE"],
    marker_color=sc,
    marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.8,
    text=[f"{v:.3f} kW" for v in s_err["MAE"]],
    textposition="outside",
    textfont=dict(size=12, color="#2C2C2C"),
))
fig.update_layout(
    title=f"MAE by Season — Summer Easiest (0.265 kW), Winter Hardest (0.361 kW) — {pct_harder:.0f}% Harder",
    title_x=0.5, height=420,
    xaxis_title="Season", yaxis_title="MAE (kW)",
    yaxis=dict(range=[0, s_err["MAE"].max() * 1.22]),
)
fig.show()

## 16. Quantile Performance Analysis

**Why:** Aggregate metrics like RMSE hide performance at different consumption levels. A model may be excellent for mid-range loads but poor at extremes — which matters most for demand-response systems.

**What we find (from actual quantile chart):**
| Quantile | Range | MAE | Interpretation |
|---|---|---|---|
| Q0–25 | Low loads | 0.182 kW | Small absolute error but high relative error (standby loads are tiny) |
| Q25–50 | Below median | 0.276 kW | Model accuracy improving with consumption level |
| Q50–75 | Above median | 0.307 kW | Mid-range — best balance of absolute and relative error |
| Q75–90 | High loads | 0.330 kW | Still good; model captures peak-adjacent patterns |
| **Q90–100** | **Extreme peaks** | **0.751 kW** | **MAE more than doubles** — rare simultaneous appliance events are hardest |

**Key implication:** The model is weakest exactly where it matters most for grid management — at peak consumption events (> Q90). Adding real-time temperature or occupancy signals would most improve this highest-error regime.

In [ ]:
quantiles = [0, 0.25, 0.5, 0.75, 0.9, 1.0]
labels    = ["Q0–25", "Q25–50", "Q50–75", "Q75–90", "Q90–100"]
bins      = pd.qcut(error_df["actual"], q=quantiles, labels=labels)
q_err     = error_df.groupby(bins)["abs_err"].agg(["mean","median","count"])
q_err.columns = ["MAE","Median_AE","Count"]
q_err["Range_kW"] = [
    f"{error_df['actual'].quantile(quantiles[i]):.2f}–{error_df['actual'].quantile(quantiles[i+1]):.2f}"
    for i in range(len(labels))
]
print("Error by consumption quantile:")
print(q_err.to_string())

q_colors = ["#2E7D32","#1565C0","#1565C0","#E65100","#C62828"]

fig = go.Figure(go.Bar(
    x=q_err.index.astype(str), y=q_err["MAE"],
    marker_color=q_colors,
    marker_line_color="rgba(0,0,0,0.1)", marker_line_width=0.7,
    text=[f"{v:.3f} kW" for v in q_err["MAE"]],
    textposition="outside",
    textfont=dict(size=11),
))
fig.add_annotation(
    x="Q90–100", y=q_err["MAE"].max() + 0.04,
    text=f"2.3× worse than Q50–75",
    showarrow=True, arrowhead=2, arrowcolor="#C62828", ax=0, ay=-40,
    font=dict(size=10, color="#C62828"),
)
fig.update_layout(
    title="MAE by Consumption Quantile — Extreme Peaks (Q90–100) Have 4× the Error of Low Loads",
    title_x=0.5, height=440,
    xaxis_title="Consumption Quantile", yaxis_title="MAE (kW)",
    yaxis=dict(range=[0, q_err["MAE"].max() * 1.22]),
    annotations=[dict(
        text="Green = easiest  |  Blue = mid-range  |  Red = hardest (peak events)",
        x=0.5, y=1.09, xref="paper", yref="paper",
        showarrow=False, font=dict(size=10, color="#555"), xanchor="center",
    )],
)
fig.show()

## 17. Save Models & Feature Columns


In [ ]:
# Save feature columns for use by the Streamlit app
import json
with open("../models/feature_cols.json","w") as f:
    json.dump(feature_cols, f)

print("✅ All models saved to ../models/")
print("   ridge_pipeline.pkl | random_forest.pkl")
print("   xgboost_model.pkl  | lightgbm_model.pkl")
print("   lstm_model.keras   | feature_cols.json")


## 18. Final Conclusions & Recommendations

### Results Summary (from actual run output)

| Model | RMSE | MAE | R² |
|---|---|---|---|
| **LightGBM** ← deploy | **0.459** | **0.314** | **0.603** |
| XGBoost | 0.459 | 0.316 | 0.602 |
| Random Forest | 0.461 | 0.316 | 0.599 |
| Ridge Regression | 0.495 | 0.349 | 0.538 |
| LSTM (multivariate) | 0.584 | 0.422 | 0.357 |
| Naïve (lag-24h) | 0.778 | 0.520 | −0.140 |

**41% RMSE reduction** vs naïve baseline. **80%+ of predictions within ±0.5 kW.**

### Key Findings

- **lag_1h dominates** across all three tree models — the last hour of usage is the strongest single predictor (RF importance: 0.424, >5× the next feature)
- **LightGBM ≈ XGBoost ≈ Random Forest** (< 0.5% RMSE gap) — gradient boosting models have reached this dataset's noise floor together
- **LSTM is the worst ML model** (RMSE 0.584, R² 0.357); the 47 engineered features already encode all temporal structure the LSTM would try to learn via sequence memory
- **R² = 0.603** means ~40% of variance is unexplained — primarily unobserved occupancy and weather

### Why It's Not Better

No weather data, no occupancy signal. These two factors alone likely account for the majority of the residual 0.46 kW RMSE. Adding hourly temperature could plausibly improve R² by 15–20%.

### Actionable Recommendations

- **Evening (19–22h) forecasts carry the highest uncertainty** — hour 22 MAE = 0.49 kW; add ±30% buffer
- **Winter is ~36% harder to predict than summer** (0.361 vs 0.265 kW MAE) — widen prediction intervals seasonally
- **Peak consumption events (Q90–100) have 4× the error of low loads** — temperature/occupancy data would most improve this regime
- **If usage is high right now, expect it to remain high** — lag_1h is the dominant feature; shift discretionary loads immediately